# Demo: orquestador multi-agente

Un **supervisor** coordina a tres agentes: el **investigador** (busca en la base de productos), el **analista** (valida y calcula) y el **sintetizador** (redacta la respuesta). Los especialistas nunca hablan entre sí: todo pasa por el supervisor y por un estado compartido donde cada aporte queda con el nombre de quien lo hizo.

Este notebook muestra el flujo de delegación con el modelo real.

In [1]:
from dotenv import load_dotenv

from agents.supervisor import Limits
from config import Settings
from graph import Models, build_graph, build_model
from main import DEFAULT_QUESTION, print_flow
from runner import run_question

load_dotenv()
settings = Settings.from_env()
graph = build_graph(
    Models.same(build_model(settings)),
    Limits(max_steps=settings.max_steps, max_attempts=settings.max_attempts),
)
print(f"Modelo: {settings.provider}:{settings.model}")
print(f"Límites: {settings.max_steps} decisiones del supervisor, {settings.max_attempts} intentos por agente")

Modelo: anthropic:claude-opus-5-5
Límites: 8 decisiones del supervisor, 2 intentos por agente


## 1. Una consulta que necesita dos dominios

Hay que **investigar** (reseñas y fichas de dos productos), **analizar** (sentimiento y puntaje promedio) y **sintetizar** una recomendación. Cada `[supervisor #n]` es una decisión: a quién llama y por qué.

In [2]:
result = await run_question(graph, DEFAULT_QUESTION, recursion_limit=settings.recursion_limit)
print_flow(result)


Pregunta: Quiero comprar auriculares o un parlante. Investigá las reseñas de los Auriculares Aurora X y del Parlante Brisa Mini, analizá el sentimiento y el puntaje promedio de cada uno y recomendame cuál conviene.

[supervisor #1] -> investigador
    razón: Todavía no hay aportes. Según el flujo esperado, primero el investigador tiene que reunir las reseñas, los puntajes y las fichas de ambos productos. Sin esos datos, el analista no puede calcular el sentimiento ni el puntaje promedio, y el sintetizador no puede recomendar.
    instrucción: Buscá en la base de productos las reseñas de los Auriculares Aurora X y del Parlante Brisa Mini. Para cada producto devolvé: el texto completo de cada reseña, el puntaje individual de cada una (por ejemplo, de 1 a 5) y la cantidad total de reseñas. Incluí también el precio y los datos principales de la ficha de cada producto, que sirven de contexto para la recomendación. Todos los datos tienen que provenir de las herramientas de búsqueda, sin agr

## 2. El estado compartido: quién aportó qué

Cada aporte queda atribuido a su agente, con los datos crudos de sus herramientas. El analista recibe los datos exactos del investigador, no un resumen.

In [3]:
for aporte in result["contribuciones"]:
    herramientas = [dato["herramienta"] for dato in aporte["datos"]]
    print(f"{aporte['agente']:<13} intento {aporte['intento']} | herramientas: {herramientas or '-'}")

investigador  intento 1 | herramientas: ['buscar_resenas', 'buscar_resenas', 'buscar_ficha_producto', 'buscar_ficha_producto']
analista      intento 1 | herramientas: ['validar_resenas', 'validar_resenas', 'analizar_sentimiento', 'analizar_sentimiento', 'calcular_estadisticas', 'calcular_estadisticas']
sintetizador  intento 1 | herramientas: -


## 3. Cuando falta información

Si se pide un producto que no existe, el investigador lo detecta con sus herramientas y el supervisor sigue el flujo en vez de inventar datos. El informe final dice que el producto no está en la base.

In [4]:
pregunta = "Compará las reseñas del Teclado Forja K1 con las del Mouse Zafiro y decime cuál recomendás."
resultado_2 = await run_question(graph, pregunta, recursion_limit=settings.recursion_limit)
print_flow(resultado_2)


Pregunta: Compará las reseñas del Teclado Forja K1 con las del Mouse Zafiro y decime cuál recomendás.

[supervisor #1] -> investigador
    razón: No hay aportes todavía. Según el flujo esperado, primero el investigador tiene que reunir las reseñas y los datos de ambos productos. Sin eso, el analista no puede calcular sentimiento ni estadísticas y la recomendación no tendría evidencia (criterios de completitud y evidencia).
    instrucción: Buscá en la base de productos las reseñas del Teclado Forja K1 y del Mouse Zafiro. Para cada producto devolvé: nombre exacto según la ficha, precio actual, cantidad total de reseñas, calificación promedio (si está disponible) y el texto completo de las reseñas (o una muestra representativa de al menos 10 por producto si son muchas, indicando cuántas hay en total). Incluí también los datos clave de cada ficha técnica. Citá la herramienta o fuente de la que sale cada dato y no inventes ni estimes cifras. Si alguno de los dos productos no aparece o no 

## 4. Topología del grafo

El mismo diagrama Mermaid que está en el README.

In [5]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	investigador(investigador)
	analista(analista)
	sintetizador(sintetizador)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analista --> supervisor;
	investigador --> supervisor;
	sintetizador --> supervisor;
	supervisor -. &nbsp;FINISH&nbsp; .-> __end__;
	supervisor -.-> analista;
	supervisor -.-> investigador;
	supervisor -.-> sintetizador;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

